# Classificação de Sentimentos: Teorema de Bayes e TF-IDF

O Naive Bayes aplica o Teorema de Bayes assumindo independência entre as características (palavras). A fórmula fundamental é:
$$P(y|x_1,...,x_n) = \frac{P(y)P(x_1,...,x_n|y)}{P(x_1,...,x_n)}$$

## Exemplo 1: Pipeline Profissional

In [ ]:
# 1. Importação dos módulos padrão da indústria
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# 2. Toy Dataset: Dados mínimos para rastreio matemático
reviews = [
    "O produto é excelente, entrega rápida", # Classe 1 (Positivo)
    "Qualidade muito boa e bonito",          # Classe 1 (Positivo)
    "Péssimo produto, veio quebrado",        # Classe 0 (Negativo)
    "Odiei a experiência, suporte ruim"      # Classe 0 (Negativo)
]
sentimentos = [1, 1, 0, 0]

# 3. Construção do Pipeline (Industry Standard)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

# 4. Treinamento e Predição
pipeline.fit(reviews, sentimentos)

# Testando uma frase com palavra de forte verossimilhança negativa
nova_frase = ["O suporte é horrível e o produto quebrado"]
predicao = pipeline.predict(nova_frase)

print(f"Review: {nova_frase[0]}")
print(f"Classe Predita: {'Positivo' if predicao[0] == 1 else 'Negativo'}")

## Exemplo 2: Mecânica passo a passo e Integração com Pandas

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report

# Dataset minimalista
mensagens = [
    "entrega muito rápida produto ótimo", # 1
    "atrasou demais e veio quebrado",     # 0
    "excelente custo benefício",          # 1
    "péssimo atendimento e demora",       # 0
    "recomendo a todos, muito bom"        # 1
]
labels = [1, 0, 1, 0, 1]

# Vetorização BoW
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(mensagens)

# Modelo simples
modelo_simples = MultinomialNB()
modelo_simples.fit(X, labels)

# Teste com frase inédita
teste = ["entrega rápida e produto bom"]
X_teste = vectorizer.transform(teste)
predicao_simples = modelo_simples.predict(X_teste)
print(f"Teste Simples (BoW): Classe Predita: {'Positivo' if predicao_simples[0] == 1 else 'Negativo'}\n")

# --- INTEGRAÇÃO COM TF-IDF E PANDAS ---
dados = {
    'review': ["O produto chegou antes do prazo", "Não gostei, veio com defeito", 
               "Muito bom, recomendo!", "Péssima qualidade, solicitei devolução"],
    'sentimento': [1, 0, 1, 0]
}
df = pd.DataFrame(dados)

# Divisão
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df['review'], df['sentimento'], test_size=0.3, random_state=42)

# Vetorização e Classificação
tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

classificador = MultinomialNB()
classificador.fit(X_train, y_train)
y_pred = classificador.predict(X_test)

print("Relatório de Classificação (TF-IDF):")
print(classification_report(y_test, y_pred, zero_division=0))

## Exemplo 3: Interpretabilidade

In [ ]:
import numpy as np

def explicar_sentimento(modelo, vetorizador):
    features = vetorizador.get_feature_names_out()
    
    for i, classe in enumerate(['Negativo', 'Positivo']):
        # Ordena pelos log-probs mais altos
        top_indices = modelo.feature_log_prob_[i].argsort()[-5:][::-1]
        print(f"\nPalavras mais fortes para {classe}:")
        for idx in top_indices:
            print(f" -> {features[idx]}")

explicar_sentimento(classificador, tfidf)